In [1]:
# Set Up Library
# Library untuk PATH
from pathlib import Path

# Library FFMPEG dan progress bar (folder and files management)
import subprocess
from tqdm import tqdm
import shutil
import os

# Library Mediapipe Blazepose
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# Library OpenCV-Python
import cv2 as cv

# Library base
import numpy as np

### **Data Pre-Processing for Kinetics Dataset**

**1: Video Data Standardization**

In [2]:
# Read PATH kinetics_dataset
data_dir = Path('kinetics_dataset')

classes = [d for d in data_dir.iterdir() if d.is_dir()]
print(classes)

[WindowsPath('kinetics_dataset/jumping_jacks'), WindowsPath('kinetics_dataset/knee_push_up'), WindowsPath('kinetics_dataset/mountain_climber'), WindowsPath('kinetics_dataset/sit_up'), WindowsPath('kinetics_dataset/squat')]


In [5]:
# Ambil semua file video dari setiap kelas
vid_extensions = [".mp4", ".webm", ".mkv"]

vid_files = []

for cls_folder in classes:
  for file in cls_folder.iterdir():
    if file.suffix.lower() in vid_extensions:
      vid_files.append(file)

print(f"Total video: {len(vid_files)}")

Total video: 500


In [11]:
# Setup output folder hasil data cleaning
output_dir = Path('kinetics_dataset_standardized')
output_dir.mkdir(exist_ok=True)

In [7]:
# Konversi video ke format .mp4 dan 30 fps dengan FFMPEG
def run(cmd):
    result = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    return result.returncode == 0

for cls_folder in data_dir.iterdir():
    if not cls_folder.is_dir():
        continue

    out_cls = output_dir / cls_folder.name
    out_cls.mkdir(parents=True, exist_ok=True)

    for video in cls_folder.iterdir():
        if video.suffix.lower() not in [".mp4", ".webm", ".mkv"]:
            continue

        output_file = out_cls / (video.stem + ".mp4")

        print(f"\nMengkonversi video: {video.name}")

        # Attempt 1 (normal mode)
        cmd1 = [
            "ffmpeg", "-y",
            "-i", str(video),
            "-r", "30",
            "-c:v", "libx264",
            "-c:a", "aac",
            str(output_file)
        ]

        if run(cmd1):
            continue

        print("Normal mode gagal konversi, mencoba safe mode")

        # Attempt 2 (safe mode)
        cmd2 = [
            "ffmpeg", "-y",
            "-fflags", "+genpts",
            "-err_detect", "ignore_err",
            "-i", str(video),
            "-r", "30",
            "-c:v", "libx264",
            "-c:a", "aac",
            str(output_file)
        ]

        if run(cmd2):
            continue

        print("Safe mode gagal konversi, mencoba force decode")

        # Attempt 3 (force decode)
        cmd3 = [
            "ffmpeg", "-y",
            "-err_detect", "ignore_err",
            "-i", str(video),
            "-vf", "fps=30",
            "-an",
            "-c:v", "libx264",
            str(output_file)
        ]

        if run(cmd3):
            continue

        print("Percobaan terakhir (frame rebuild)")

        temp_dir = output_dir / "temp_frames"
        temp_dir.mkdir(exist_ok=True)

        # extract frame
        cmd4 = [
            "ffmpeg", "-y",
            "-err_detect", "ignore_err",
            "-i", str(video),
            str(temp_dir / "frame_%04d.jpg")
        ]

        if run(cmd4):
            cmd5 = [
                "ffmpeg", "-y",
                "-r", "30",
                "-i", str(temp_dir / "frame_%04d.jpg"),
                "-c:v", "libx264",
                str(output_file)
            ]
            if run(cmd5):
                # cleanup frame
                for f in temp_dir.glob("*.jpg"):
                    f.unlink()
                continue

        print(f"Gagal mengkonversi: {video.name}")


Mengkonversi video: 1cIiXGc8ahw.mkv

Mengkonversi video: 1hJ5gTmwnqg.mp4

Mengkonversi video: 1KvnF-2atZI.mp4

Mengkonversi video: 1UrOZKaJIXk.webm

Mengkonversi video: 2LonkCR8yVU.mkv

Mengkonversi video: 2pKaMhmF3zw.mkv

Mengkonversi video: 2WH9wMcD9EY.mp4

Mengkonversi video: 2ymRogDlpdk.mkv

Mengkonversi video: 3LhI0O6H6Oo.mp4

Mengkonversi video: 3Uma_5bRzCg.webm

Mengkonversi video: 4-cWfJa7H8k.mp4

Mengkonversi video: 4hcR5Sfmnbk.webm

Mengkonversi video: 4WmMcx53Zwo.mkv

Mengkonversi video: 4XDZFKFp7QQ.mkv

Mengkonversi video: 4Z5e3KdOiqI.mkv

Mengkonversi video: 5Hn3btb3ZY8.webm

Mengkonversi video: 5htwPbyi5GM.mkv

Mengkonversi video: 5hWI-xxbJzA.mkv

Mengkonversi video: 5tg3lWCaljc.mp4

Mengkonversi video: 60kf07PNq9E.mp4

Mengkonversi video: 6ktL_l51eUo.mp4

Mengkonversi video: 6LJl-pDtDnc.mkv

Mengkonversi video: 6SQJfkIsKr0.mp4

Mengkonversi video: 6tfU7s3puQ4.webm

Mengkonversi video: 75eBXz9V0-A.webm

Mengkonversi video: 7GtJZeuy8gc.webm

Mengkonversi video: 7n7PNQU-6A

In [ ]:
# Menghapus folder temp frames jika masih ada
if Path("kinetics_dataset_cleaned/temp_frames").exists():
  shutil.rmtree("kinetics_dataset_cleaned/temp_frames")

**2: Pose-Based Data Extraction**

In [ ]:
# Setup Mediapipe Blazepose Full .task
mediapipe_path = "pose_landmarker_full.task"

def create_landmarker():
  base_options = python.BaseOptions(model_asset_path=mediapipe_path)

  options = vision.PoseLandmarkerOptions(
        base_options=base_options,
        running_mode=vision.RunningMode.VIDEO,
        num_poses=1
    )
  
  return vision.PoseLandmarker.create_from_options(options)

In [13]:
# Read PATH kinetics_dataset_cleaned
data_cleaned_dir = Path('kinetics_dataset_standardized')

classes = [d for d in data_cleaned_dir.iterdir() if d.is_dir()]
print(classes)

[WindowsPath('kinetics_dataset_standardized/jumping_jacks'), WindowsPath('kinetics_dataset_standardized/knee_push_up'), WindowsPath('kinetics_dataset_standardized/mountain_climber'), WindowsPath('kinetics_dataset_standardized/sit_up'), WindowsPath('kinetics_dataset_standardized/squat')]


In [14]:
# Fungsi ekstraksi pose dari frame video yang sudah distandardisasi
def extract_pose_from_video(video_path, pose_landmarker):
  capture = cv.VideoCapture(str(video_path))

  fps = capture.get(cv.CAP_PROP_FPS)
  if not fps or fps != fps:
    fps = 30
  
  frame_idx = 0
  prev_timestamp = -1

  frames = []

  while True:
    ret, frame = capture.read()
    if not ret:
      break

    timestamps_ms = int(frame_idx * (1000.0 / fps))
    frame_idx += 1

    frame_rgb = cv.cvtColor(frame, cv.COLOR_BGR2RGB)

    mp_image = mp.Image(
      image_format=mp.ImageFormat.SRGB, 
      data=frame_rgb)

    result = pose_landmarker.detect_for_video(mp_image, timestamps_ms)

    if result.pose_landmarks:
      landmarks = result.pose_landmarks[0]
      frames.append([[lm.x, lm.y, lm.z] for lm in landmarks])
    else:
      frames.append(np.zeros((33, 3)))

  capture.release()

  return np.array(frames), fps  

In [ ]:
# Loop fungsi ekstraksi pose ke semua video
seq_output_dir = Path('pose_sequences_kinetics')
seq_output_dir.mkdir(exist_ok=True)

for cls_folder in data_cleaned_dir.iterdir():
  if not cls_folder.is_dir():
    continue

  out_cls = seq_output_dir / cls_folder.name
  out_cls.mkdir(parents=True, exist_ok=True)

  for video_path in cls_folder.glob("*.mp4"):
    print(f"Memproses output sekuens: {video_path.name}")

    pose_landmarker = create_landmarker()
    
    seq, fps = extract_pose_from_video(video_path, pose_landmarker)

    np.save(out_cls / f"{video_path.stem}.npy", seq)

Memproses output sekuens: 1cIiXGc8ahw.mp4
Memproses output sekuens: 1hJ5gTmwnqg.mp4
Memproses output sekuens: 1KvnF-2atZI.mp4
Memproses output sekuens: 1UrOZKaJIXk.mp4
Memproses output sekuens: 2LonkCR8yVU.mp4
Memproses output sekuens: 2pKaMhmF3zw.mp4
Memproses output sekuens: 2WH9wMcD9EY.mp4
Memproses output sekuens: 2ymRogDlpdk.mp4
Memproses output sekuens: 3LhI0O6H6Oo.mp4
Memproses output sekuens: 3Uma_5bRzCg.mp4
Memproses output sekuens: 4-cWfJa7H8k.mp4
Memproses output sekuens: 4hcR5Sfmnbk.mp4
Memproses output sekuens: 4WmMcx53Zwo.mp4
Memproses output sekuens: 4XDZFKFp7QQ.mp4
Memproses output sekuens: 4Z5e3KdOiqI.mp4
Memproses output sekuens: 5Hn3btb3ZY8.mp4
Memproses output sekuens: 5htwPbyi5GM.mp4
Memproses output sekuens: 5hWI-xxbJzA.mp4
Memproses output sekuens: 5tg3lWCaljc.mp4
Memproses output sekuens: 60kf07PNq9E.mp4
Memproses output sekuens: 6ktL_l51eUo.mp4
Memproses output sekuens: 6LJl-pDtDnc.mp4
Memproses output sekuens: 6SQJfkIsKr0.mp4
Memproses output sekuens: 6tfU7s3p

In [2]:
# Melihat contoh data sekuens pose
example_data1 = np.load("pose_sequences_kinetics/squat/_m4TDbpIQPI.npy")

print("shape:", example_data1.shape)
print("dtype:", example_data1.dtype)

print("\nframe pertama:\n", example_data1[0])
print("\nframe kedua:\n", example_data1[1])

# mengecek jumlah frame koson
zero_frames = np.sum(np.all(example_data1 == 0, axis=(1,2)))
persentase_zero = ((zero_frames / len(example_data1)) * 100)
print("\nframe kosong:", zero_frames, "/", len(example_data1), f"({persentase_zero:.2f}%)")

shape: (513, 33, 3)
dtype: float64

frame pertama:
 [[ 0.31889179  0.49322012 -0.01312189]
 [ 0.31786603  0.4768827  -0.03825476]
 [ 0.31859982  0.47639257 -0.03825202]
 [ 0.31933463  0.4759365  -0.03829477]
 [ 0.31913978  0.47429267 -0.00773006]
 [ 0.32065862  0.47213709 -0.00771985]
 [ 0.3222287   0.46989322 -0.00777977]
 [ 0.33395377  0.46916178 -0.1220674 ]
 [ 0.33829588  0.45916012  0.01561918]
 [ 0.33015621  0.49987668 -0.04126952]
 [ 0.33082384  0.4990986  -0.00142318]
 [ 0.38378564  0.52033192 -0.19982789]
 [ 0.39330444  0.48289338  0.07871304]
 [ 0.48124102  0.54085875 -0.20564881]
 [ 0.47619009  0.5150001   0.1287231 ]
 [ 0.53477788  0.54378235 -0.19978966]
 [ 0.5216527   0.53299052  0.11087584]
 [ 0.54584026  0.54440761 -0.22977386]
 [ 0.53898293  0.54020303  0.11298151]
 [ 0.54368848  0.54102761 -0.23302718]
 [ 0.53423667  0.52896041  0.09484685]
 [ 0.54215825  0.5408901  -0.20256266]
 [ 0.52774215  0.53588098  0.1014111 ]
 [ 0.491889    0.67516357 -0.10472098]
 [ 0.4934305

In [3]:
# Melihat contoh data sekuens pose #2
example_data2 = np.load("pose_sequences_kinetics/knee_push_up/2V6zHS9McGE.npy")

print("shape:", example_data2.shape)
print("dtype:", example_data2.dtype)

print("\nframe pertama:\n", example_data2[0])
print("\nframe kedua:\n", example_data2[1])

# mengecek jumlah frame koson
zero_frames = np.sum(np.all(example_data2 == 0, axis=(1,2)))
persentase_zero = ((zero_frames / len(example_data2)) * 100)
print("\nframe kosong:", zero_frames, "/", len(example_data2), f"({persentase_zero:.2f}%)")

shape: (302, 33, 3)
dtype: float64

frame pertama:
 [[ 0.54085135  0.54687726 -0.52549452]
 [ 0.54457152  0.54040891 -0.53651053]
 [ 0.54665452  0.53838837 -0.53648967]
 [ 0.54931289  0.5353443  -0.53655022]
 [ 0.53708398  0.54068756 -0.54585034]
 [ 0.53510344  0.53974289 -0.54599243]
 [ 0.53165859  0.53745383 -0.54606926]
 [ 0.55308831  0.5158872  -0.48237309]
 [ 0.52328587  0.51620114 -0.52398807]
 [ 0.54335189  0.54321903 -0.49209765]
 [ 0.53610045  0.54413188 -0.50453752]
 [ 0.57074982  0.4869934  -0.36563113]
 [ 0.49059385  0.47187132 -0.425612  ]
 [ 0.5776841   0.54817444 -0.25371042]
 [ 0.45432273  0.54344624 -0.40827009]
 [ 0.56609672  0.63245887 -0.24218442]
 [ 0.47416484  0.6352644  -0.44704008]
 [ 0.56831211  0.65111214 -0.26325083]
 [ 0.47208744  0.65899003 -0.49273598]
 [ 0.56257623  0.64996016 -0.28108415]
 [ 0.48533085  0.65793145 -0.4974556 ]
 [ 0.55983281  0.64142925 -0.25037971]
 [ 0.48707563  0.65034211 -0.45104042]
 [ 0.53502047  0.52564424  0.01312147]
 [ 0.4890510

**3: Sequence Interpolation**

Untuk sekuens pose yang memiliki frame kosong pada threshold 5% - 25%

In [4]:
# Fungsi untuk mengecek frame kosong dari sebuah video
def check_sequence_quality(seq):
  total_frames = len(seq)
  zero_frames = np.sum(np.all(seq == 0, axis=(1,2)))
  ratio = zero_frames / total_frames

  return ratio

In [7]:
# Fungsi untuk menginterpolasi frame kosong (Linear Interpolation)
def fill_zero_frames(seq):
    seq = seq.copy()
    T = len(seq)
    zero_mask = np.all(seq == 0, axis=(1, 2))
    valid_idx = np.where(~zero_mask)[0]

    if len(valid_idx) == 0:
        return seq

    for i in np.where(zero_mask)[0]:
        before = valid_idx[valid_idx < i]
        after = valid_idx[valid_idx > i]

        if len(before) > 0 and len(after) > 0:
            i0, i1 = before[-1], after[0]
            alpha = (i - i0) / (i1 - i0)
            seq[i] = (1 - alpha) * seq[i0] + alpha * seq[i1]
        elif len(before) > 0:
            seq[i] = seq[before[-1]]
        else:
            seq[i] = seq[after[0]]
    return seq 

In [8]:
# Fungsi untuk memproses interpolasi
def interpolate_zero_frames(seq):
  ratio = check_sequence_quality(seq)

  if ratio < 0.05:
    return seq
  elif ratio < 0.25:
    print(f"Interpolasi frame kosong dengan persentase: {ratio:.2%}")
    return fill_zero_frames(seq)
  else:
    return None

In [9]:
# Read PATH pose_sequences untuk kinetics data
pose_seq = Path('pose_sequences_kinetics')

classes = [d for d in pose_seq.iterdir() if d.is_dir()]
print(classes)

[WindowsPath('pose_sequences_kinetics/jumping_jacks'), WindowsPath('pose_sequences_kinetics/knee_push_up'), WindowsPath('pose_sequences_kinetics/mountain_climber'), WindowsPath('pose_sequences_kinetics/sit_up'), WindowsPath('pose_sequences_kinetics/squat')]


In [10]:
# Apply fungsi interpolasi pada frame kosong dari video
zero_frame_exist = False

for file in pose_seq.rglob("*.npy"):
  seq = np.load(file)

  if np.any(np.all(seq == 0, axis=(1, 2))):
    zero_frame_exist = True

  seq_clean = interpolate_zero_frames(seq)

  if seq_clean is None:
    print(f"Drop sekuens pose dari video: {file.name}")
    os.remove(file)
    continue

  np.save(file, seq_clean)

if not zero_frame_exist:
  print("Tidak ada frame video yang kosong (zero)")


Interpolasi frame kosong dengan persentase: 21.36%
Interpolasi frame kosong dengan persentase: 6.64%
Interpolasi frame kosong dengan persentase: 24.24%
Interpolasi frame kosong dengan persentase: 9.93%
Interpolasi frame kosong dengan persentase: 7.14%
Interpolasi frame kosong dengan persentase: 5.10%
Drop sekuens pose dari video: 6LJl-pDtDnc.npy
Drop sekuens pose dari video: 8IvONouOs68.npy
Drop sekuens pose dari video: 9zBLYVBqtsc.npy
Interpolasi frame kosong dengan persentase: 19.00%
Interpolasi frame kosong dengan persentase: 9.16%
Interpolasi frame kosong dengan persentase: 11.91%
Drop sekuens pose dari video: ongGj_0gZRc.npy
Drop sekuens pose dari video: _ICI92IpGFc.npy
Drop sekuens pose dari video: _ptvEX38SkY.npy
Interpolasi frame kosong dengan persentase: 8.31%
Interpolasi frame kosong dengan persentase: 13.25%
Interpolasi frame kosong dengan persentase: 20.55%
Interpolasi frame kosong dengan persentase: 8.25%
Interpolasi frame kosong dengan persentase: 20.27%
Interpolasi frame